In [44]:
import numpy as np
import pandas as pd

# Генерація даних для Варіанта 1 (Київ)
np.random.seed(42)
base_temp = 9.5
amplitude = 14
city = "Київ"
rows = []

for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)  # пік влітку (місяць 7)
        noise = np.random.normal(0, 1.0)
        rows.append({
            "місто": city,
            "рік": year,
            "місяць": month,
            "температура": round(base_temp + seasonal + noise, 1),
        })

climate = pd.DataFrame(rows)
print("Перші 5 рядків довгого набору (climate):")
display(climate.head())

# Завдання 1. groupby і agg за роками
year_agg = climate.groupby("рік")["температура"].agg(["mean", "min", "max"])
print("\nАгрегація за роками:")
display(year_agg)

Перші 5 рядків довгого набору (climate):


,місто,рік,місяць,температура
0,Київ,2021,1,-4.0
1,Київ,2021,2,-2.8
2,Київ,2021,3,3.1
3,Київ,2021,4,11.0
4,Київ,2021,5,16.3



Агрегація за роками:


,mean,min,max
рік,,,
2021,9.783333,-4.0,25.1
2022,8.916667,-4.5,22.6
2023,9.308333,-5.0,23.5
2024,9.158333,-4.6,23.4


In [45]:
# Завдання 2. groupby і agg за місяцями
month_agg = climate.groupby("місяць")["температура"].agg(["mean", "std"])
print("Агрегація за місяцями:")
display(month_agg)

Агрегація за місяцями:


,mean,std
місяць,,
1,-4.400,0.424264
2,-3.600,1.104536
3,1.600,1.023067
4,9.875,0.865544
5,16.225,0.727438
6,21.600,0.294392
7,23.500,1.116542
8,21.850,1.420094
9,16.375,1.250000


In [46]:
# Завдання 3. pivot_table
climate_pivot = climate.pivot_table(index="місяць", columns="рік", values="температура", aggfunc="mean")
print("Зведена таблиця (місяці х роки):")
display(climate_pivot)

Зведена таблиця (місяці х роки):


рік,2021,2022,2023,2024
місяць,,,,
1,-4.0,-4.3,-5.0,-4.3
2,-2.8,-4.5,-2.5,-4.6
3,3.1,0.8,1.3,1.2
4,11.0,8.9,9.9,9.7
5,16.3,15.5,15.9,17.2
6,21.4,21.9,21.3,21.8
7,25.1,22.6,22.9,23.4
8,22.4,20.2,23.5,21.3
9,16.0,18.0,16.5,15.0


In [47]:
# Завдання 4. Похідні категорії і crosstab
def get_season(month):
    if month in [12, 1, 2]:
        return "зима"
    elif month in [3, 4, 5]:
        return "весна"
    elif month in [6, 7, 8]:
        return "літо"
    else:
        return "осінь"

climate["сезон"] = climate["місяць"].apply(get_season)
climate["тепліше_за_середнє"] = climate["температура"] > base_temp

# Побудова crosstab
cross_res = pd.crosstab(climate["сезон"], climate["тепліше_за_середнє"])
print("Перехресна таблиця частот (crosstab):")
display(cross_res)

Перехресна таблиця частот (crosstab):


тепліше_за_середнє,False,True
сезон,,
весна,5,7
зима,12,0
літо,0,12
осінь,7,5


In [48]:
# Завдання 5. Виклик pivot() без aggfunc
try:
    pivoted_direct = climate.pivot(index="місяць", columns="рік", values="температура")
    print("Метод pivot() спрацював успішно!")
    display(pivoted_direct.head())
except Exception as e:
    print("Помилка:", e)

Метод pivot() спрацював успішно!


рік,2021,2022,2023,2024
місяць,,,,
1,-4.0,-4.3,-5.0,-4.3
2,-2.8,-4.5,-2.5,-4.6
3,3.1,0.8,1.3,1.2
4,11.0,8.9,9.9,9.7
5,16.3,15.5,15.9,17.2


### Відповіді на контрольні запитання:

1. **Логіка парадигми split-apply-combine у groupby():**
   - **Split (Розбиття):** вихідний DataFrame розбивається на менші групи на основі унікальних значень у ключовому стовпці (наприклад, за роком чи місяцем).
   - **Apply (Застосування):** до кожної окремої групи незалежно застосовується агрегаційна функція чи обчислення (наприклад, `.mean()` чи `.agg()`).
   - **Combine (Об'єднання):** результати обчислень для всіх груп збираються назад в єдину зведену структуру чи серію.

2. **Відмінність між pivot() і pivot_table():**
   - `pivot()` виконує просте переформатування таблиці без агрегації і ламається з помилкою, якщо для якоїсь комбінації індекс/стовпець існує більше одного значення (дублікати). `pivot_table()` є потужнішим аналогом, який автоматично обробляє дублікати завдяки явному чи неявному параметру агрегації (`aggfunc`), зводячи кілька значень до одного.

3. **Що показує crosstab() і чим відрізняється від groupby(...).size():**
   - `crosstab()` будує матрицю частот для двох або більше категоріальних змінних (зручно для аналізу взаємного розподілу категорій). Еквівалентний результат можна отримати через `groupby([...]).size().unstack()`, проте `crosstab()` пропонує набагато простіший і компактніший інтерфейс для побудови таблиць спряженості.

4. **Чому для groupby і pivot_table потрібен довгий (tidy) формат:**
   - У довгому форматі кожна змінна міститься в окремому стовпці, а кожне спостереження — в окремому рядку. Це забезпечує універсальність: функції групування та зведення можуть орієнтуватися на чіткі імена стовпців без прив'язки до розтягнутих у ширину назв місяців чи років, запобігаючи втраті метаданих та ускладненню логіки обробки.